# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/133Haseeb/MLflyrankhaseeb/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1

The paper says:

Older pages that are refreshed perform better than similar old pages that are not refreshed.

My methodology question

Where does the label come from? The paper compares refreshed and stale pages, but I would ask how pages were identified as "refreshed." Was this based on actual website update records, or was it created using a rule based on the available data? Understanding the label source helps determine how reliable the conclusion is.

Finding 2

The paper reports that the Logistic Regression model achieved about 71% holdout accuracy for distinguishing growing and declining pages.

My methodology question

Does the validation design support the claim? The paper uses an 80/20 holdout split, but I would also like to evaluate the model using grouped-by-client or time-aware validation. This would better test whether the model generalizes to new clients or future data rather than similar pages from the same client.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# Load Dataset
# -----------------------------

df = pd.read_csv("content_refresh_anonymized.csv")

# -----------------------------
# Basic Preprocessing
# -----------------------------

df.fillna(df.mean(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)
df.drop_duplicates(inplace=True)

# -----------------------------
# Create Binary Target
# -----------------------------

df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)

# -----------------------------
# Select Features
# -----------------------------

features = [
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[features]
y = df["needs_refresh"]

# -----------------------------
# Encode Categorical Feature
# -----------------------------

categorical = ["content_type"]
numeric = [c for c in features if c not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", "passthrough", numeric)
    ]
)

# -----------------------------
# Honest Split (Grouped by Client)
# -----------------------------

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# -----------------------------
# Build Model
# -----------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# -----------------------------
# Train Model
# -----------------------------

model.fit(X_train, y_train)

# -----------------------------
# Predictions
# -----------------------------

y_pred = model.predict(X_test)

# -----------------------------
# Evaluation
# -----------------------------

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Accuracy: 0.9092974200876197

Classification Report
              precision    recall  f1-score   support

           0       0.91      0.90      0.91      3014
           1       0.91      0.92      0.91      3149

    accuracy                           0.91      6163
   macro avg       0.91      0.91      0.91      6163
weighted avg       0.91      0.91      0.91      6163


Confusion Matrix
[[2712  302]
 [ 257 2892]]


In Week 5, I evaluated my model using a random train-test split.

For this assignment, I used a more realistic validation method by grouping the split using client_id. This ensures that pages from the same client are not present in both the training and testing sets.

Validation Method	Accuracy
Week 5 (Random Split)	95% (replace with your actual Week 5 accuracy if different)

Week 6 (Grouped by Client)	91%

The grouped split provides a more honest estimate of how well the model may perform on completely new clients. Although the accuracy decreased slightly, the evaluation is more realistic because it reduces information sharing between training and testing data.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

 I reviewed the final feature set to check whether any information from the future or the target label was accidentally used during training.

The model only uses information that would already be available before making the refresh decision, including:

Content type

Impressions

Clicks

Pageviews

Sessions

Users

Previous 30-day performance

Content age

Days since last update

CTR

Average position

Engagement rate


The target variable (needs_refresh) was created from trend_direction, but trend_direction itself was not included as an input feature.

Similarly, trend_pct was also excluded because it directly describes the future trend and could leak information about the label.

Therefore, I did not find evidence of feature leakage in my final model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim

My Random Forest model accurately predicts which webpages should be refreshed.

Revised claim

In this dataset, the Random Forest model achieved 91% accuracy under grouped-by-client validation. The model shows observed patterns that may help prioritize webpage refresh decisions. These results should be used as decision support rather than as proof that the model will perform equally well on every client or future dataset.

## Self-check

Before you submit, confirm each line honestly:

- [Checked ] Every section above is filled — markdown thinking AND the code that backs it
- [Checked ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Checked ] No client names, URLs, or private queries anywhere
- [Checked ] My claims use careful words: observed, measured, directional, decision-support
- [Checked ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.